# Visualize a NeqSim DEXPI engineering P&ID

This executed notebook reads either NeqSim's native DEXPI 2.0 XML or its Proteus 4.1.1 representation and creates a deterministic structural P&ID preview. It uses the same object identities and references checked by `EngineeringDexpiRoundTripQualifier`.

The figure is suitable for documentation and CI review. It is not a construction drawing and does not replace symbol, layout, connectivity, or round-trip qualification in the target CAE product.

In [1]:
from collections import Counter
from pathlib import Path
import importlib.util
import os
import sys

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pom.xml').is_file() and (path / 'examples' / 'neqsim').is_dir()
)
renderer_path = ROOT / 'examples' / 'neqsim' / 'render_engineering_pid.py'
spec = importlib.util.spec_from_file_location('render_engineering_pid', renderer_path)
renderer = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = renderer
spec.loader.exec_module(renderer)
print('Repository root located:', ROOT.name)
print('Renderer loaded:', renderer_path.relative_to(ROOT))

Repository root located: neqsim
Renderer loaded: examples/neqsim/render_engineering_pid.py


## Select a controlled XML input

Set `NEQSIM_ENGINEERING_PACKAGE` to a directory produced by `EngineeringDeliverableCompiler` to inspect your own package. The notebook otherwise uses the committed branching-process golden fixture, keeping this example reproducible without Java or network access.

In [2]:
configured_package = os.environ.get('NEQSIM_ENGINEERING_PACKAGE')
configured_xml = Path(configured_package) / 'plant.dexpi.xml' if configured_package else None
golden_xml = ROOT / 'src' / 'test' / 'resources' / 'dexpi' / '2.0' / 'golden' / 'branching-process.dexpi.xml'
if configured_xml is not None and configured_xml.is_file():
    source_xml = configured_xml
    print('Using compiler package:', source_xml)
else:
    source_xml = golden_xml
    print('Using committed golden fixture:', source_xml.relative_to(ROOT))

Using committed golden fixture: src/test/resources/dexpi/2.0/golden/branching-process.dexpi.xml


In [3]:
graph = renderer.read_pid_graph(source_xml)
categories = dict(sorted(Counter(node.category for node in graph.nodes.values()).items()))
print('Source profile:', graph.source_profile)
print('Parsed objects:', len(graph.nodes))
print('Parsed references:', len(graph.edges))
print('Categories:', categories)

Source profile: DEXPI_2_0_NATIVE
Parsed objects: 22
Parsed references: 12
Categories: {'connection': 14, 'equipment': 3, 'piping': 5}


In [4]:
output_dir = ROOT / 'build' / 'notebook-output' / 'dexpi-pid-visualization'
png_path = output_dir / 'pid-preview.png'
svg_path = output_dir / 'pid-preview.svg'
stats = renderer.render_pid(
    graph, png_path, svg_path,
    title='NeqSim native DEXPI 2.0 — structural P&ID preview',
    max_nodes=30,
)
print('Rendered PNG:', png_path.relative_to(ROOT))
print('Rendered SVG:', svg_path.relative_to(ROOT))
print('Figure statistics:', stats)

Rendered PNG: build/notebook-output/dexpi-pid-visualization/pid-preview.png
Rendered SVG: build/notebook-output/dexpi-pid-visualization/pid-preview.svg
Figure statistics: {'sourceProfile': 'DEXPI_2_0_NATIVE', 'parsedNodeCount': 22, 'parsedEdgeCount': 12, 'displayedNodeCount': 8, 'displayedEdgeCount': 6}


## Structural P&ID preview

![Rendered structural P&ID with equipment, piping objects and contracted connection nodes](figures/dexpi_engineering_pid_roundtrip.png)

Blue rounded rectangles are process equipment, red bow-tie symbols are piping components, and arrows represent direct or connection-node-contracted DEXPI references. Instrument bubbles appear when the source document contains instrumentation objects.

In [5]:
for node in list(graph.nodes.values())[:8]:
    print(f'{node.label:10} {node.category:11} {node.dexpi_type}')

10-VA-001  equipment   Plant/ProcessEquipment.Separator
INLET      connection  Plant/ProcessEquipment.Nozzle
1          connection  Plant/Piping.PipingNode
OUTLET_1   connection  Plant/ProcessEquipment.Nozzle
2          connection  Plant/Piping.PipingNode
OUTLET_2   connection  Plant/ProcessEquipment.Nozzle
3          connection  Plant/Piping.PipingNode
10-KA-001  equipment   Plant/ProcessEquipment.CentrifugalCompressor


## Use the result of the engineering compiler

Compile the Java model first:

```java
EngineeringDeliverableCompiler.CompilationResult result =
    EngineeringDeliverableCompiler.compile(project, Paths.get("build/engineering-package"));
```

Then launch Jupyter with `NEQSIM_ENGINEERING_PACKAGE=build/engineering-package`, or render directly:

```bash
python examples/neqsim/render_engineering_pid.py \
  build/engineering-package/plant.dexpi.xml \
  --png build/engineering-package/pid-preview.png \
  --svg build/engineering-package/pid-preview.svg
```

The same command accepts `plant-proteus.xml`. Use pyDEXPI or the target CAE tool for standards-aware symbols and a production drawing. Review `engineering-dexpi-roundtrip-report.json` and `engineering-validation-report.json` before any controlled issue.